# sre-gym Basic-tier evaluation — comparison table

> **The single artifact that defends the 20% "reward improvement" judging weight.**

Runs 7 policies against the 12-scenario held-out set (one `__p05` procgen variant per template) and produces:

1. A per-template / per-policy reward distribution plot (shared axes).
2. A summary comparison table (mean, median, resolved-rate per policy).
3. CSV + PNG artifacts in `eval/results/` for embedding in the README.

Policies evaluated:

| Policy | Source |
|---|---|
| Random | uniform over allowed_actions |
| Heuristic | deterministic if-else (no LLM) |
| Untrained Qwen2.5-3B | base model from HF |
| Llama-3.3-70B-Instruct | Fireworks API (or Groq fallback) |
| Claude Haiku | Anthropic API |
| Claude Sonnet | Anthropic API |
| **Trained Qwen2.5-3B (sre-gym-qwen25-3b-grpo)** | this repo's output |

The big claim: **the trained 3B beats Claude Haiku and approaches Sonnet on the held-out set.** That's the entire pitch in one cell.

## API keys required

Set in Colab Secrets:

- `ANTHROPIC_API_KEY` (for Haiku + Sonnet)
- `FIREWORKS_API_KEY` or `GROQ_API_KEY` (for Llama-3.3-70B)
- `HF_TOKEN` (to load the trained adapter)

In [ ]:
%%bash
pip install -q 'anthropic>=0.40' 'openai>=1.50' 'httpx>=0.27' 'pandas' 'matplotlib' 'pyyaml'
pip install -q 'unsloth>=2025.1.0' 'transformers>=4.46' 'peft>=0.13' 'accelerate>=1.1'
pip install -q 'openenv-core>=0.2.1' 'fastapi>=0.115' 'uvicorn[standard]>=0.30' 'pydantic>=2.8'

In [ ]:
import os, sys, json, random, time, threading
from pathlib import Path
import subprocess
REPO_ROOT = Path('/content/sre-enginnerllm')
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/dakshdoesdev/sre-enginnerllm.git', str(REPO_ROOT)])
sys.path.insert(0, str(REPO_ROOT))

import httpx
import uvicorn
import pandas as pd
import matplotlib.pyplot as plt
from unified_incident_env.server.app import create_compatible_app
from unified_incident_env.server.challenge import list_scenarios
from unified_incident_env.server.environment import UnifiedIncidentEnvironment
from unified_incident_env.models import UnifiedIncidentAction

# Boot the env in-process (no separate HTTP server needed for eval).
ALL_SCENARIOS = list_scenarios().scenarios
HOLDOUT_IDS = [s.id for s in ALL_SCENARIOS if s.id.endswith('__p05')]
print(f'Holdout: {len(HOLDOUT_IDS)} scenarios')
for sid in HOLDOUT_IDS:
    print(f'  {sid}')

## Helpers — episode runner, action parsing, observation rendering

Each policy produces a function `policy(obs) -> action_dict`. The runner steps the env, parses each action, falls back gracefully if the policy emits invalid output, and returns the final score.

In [ ]:
SYSTEM_PROMPT = '''You are a senior SRE on-call. The environment is a partially-observable
incident with 4 services (api-gateway, cache, database, worker). Use the 11 available
actions to diagnose, remediate, verify, and resolve.

Output a SINGLE JSON object on each turn:
{"action_type": "...", "service": "...", "metric": "...", "check_name": "...", "hypothesis": {...}}
Only include fields the action requires. No prose, no markdown, no code fences.'''

def parse_action(text):
    text = (text or '').strip()
    # strip code fences
    if text.startswith('```'):
        text = text.split('```', 2)[-1]
    text = text.strip()
    if not text.startswith('{'):
        i = text.find('{')
        if i == -1:
            return None
        text = text[i:]
    depth = 0; end = None
    for j, ch in enumerate(text):
        if ch == '{': depth += 1
        elif ch == '}': depth -= 1
        if depth == 0:
            end = j; break
    if end is None:
        return None
    try:
        return json.loads(text[:end+1])
    except Exception:
        return None

def run_episode(policy_fn, scenario_id, max_ticks=12):
    env = UnifiedIncidentEnvironment()
    obs = env.reset(scenario_id=scenario_id)
    ticks = 0
    while not obs.done and ticks < max_ticks:
        action_dict = policy_fn(obs)
        if action_dict is None:
            # Skip this turn with an escalate (no-op with step cost)
            action_dict = {'action_type': 'escalate'}
        try:
            action = UnifiedIncidentAction(**action_dict)
        except Exception:
            action = UnifiedIncidentAction(action_type='escalate')
        obs = env.step(action)
        ticks += 1
    return {
        'scenario_id': scenario_id,
        'final_score': float(obs.final_score),
        'incident_resolved': bool(obs.incident_resolved),
        'tick_count': int(obs.tick_count),
        'breakdown': dict(obs.score_breakdown),
    }

EPISODES_PER_SCENARIO = 3

def evaluate_policy(name, policy_fn):
    rows = []
    for sid in HOLDOUT_IDS:
        for ep in range(EPISODES_PER_SCENARIO):
            r = run_episode(policy_fn, sid)
            r['policy'] = name
            r['episode'] = ep
            r['template_id'] = sid.split('__')[0]
            rows.append(r)
    print(f'{name:<32} mean={sum(r["final_score"] for r in rows)/len(rows):.3f} resolved={sum(r["incident_resolved"] for r in rows)}/{len(rows)}')
    return rows

## Policy 1 — Random (uniform over allowed_actions)

In [ ]:
RNG = random.Random(42)
SERVICES = ['api-gateway', 'cache', 'database', 'worker']
METRICS = ['cpu', 'error_rate', 'latency']
CHECKS = ['database_recovery', 'end_to_end']

def random_policy(obs):
    actions = list(obs.allowed_actions or [])
    if not actions:
        return {'action_type': 'escalate'}
    a = RNG.choice(actions)
    if a in {'query_logs', 'query_dependencies', 'query_deploys', 'rollback_deploy', 'restart_service', 'isolate_service'}:
        return {'action_type': a, 'service': RNG.choice(SERVICES)}
    if a == 'query_metrics':
        return {'action_type': a, 'service': RNG.choice(SERVICES), 'metric': RNG.choice(METRICS)}
    if a == 'run_check':
        return {'action_type': a, 'check_name': RNG.choice(CHECKS)}
    if a == 'submit_hypothesis':
        return {'action_type': a, 'hypothesis': {
            'root_cause': 'bad_worker_deploy',
            'affected_services': ['worker'],
            'confidence': 0.5,
            'recommended_next_action': 'rollback_deploy',
        }}
    return {'action_type': a}

random_results = evaluate_policy('Random (uniform)', random_policy)

## Policy 2 — Heuristic (deterministic, no LLM)

A simple heuristic: query the loudest-alert service's logs, then its deploys, then rollback the worker, restart the database, run both checks, declare resolved. This is the dumb baseline.

In [ ]:
def heuristic_policy_factory():
    state = {'tick': 0}
    plan = [
        {'action_type': 'query_logs', 'service': 'worker'},
        {'action_type': 'query_deploys', 'service': 'worker'},
        {'action_type': 'rollback_deploy', 'service': 'worker'},
        {'action_type': 'restart_service', 'service': 'database'},
        {'action_type': 'run_check', 'check_name': 'database_recovery'},
        {'action_type': 'run_check', 'check_name': 'end_to_end'},
        {'action_type': 'declare_resolved'},
    ]
    def policy(obs):
        i = state['tick']; state['tick'] += 1
        if i < len(plan):
            return plan[i]
        return {'action_type': 'escalate'}
    return policy

def heuristic_policy(obs):
    if not hasattr(heuristic_policy, '_per_episode'):
        heuristic_policy._per_episode = {}
    sid = obs.incident_summary[:32]   # rough episode key
    if sid not in heuristic_policy._per_episode:
        heuristic_policy._per_episode[sid] = heuristic_policy_factory()
    return heuristic_policy._per_episode[sid](obs)

# Cleaner: instantiate a fresh heuristic per scenario inside evaluate_policy.
def evaluate_heuristic():
    rows = []
    for sid in HOLDOUT_IDS:
        for ep in range(EPISODES_PER_SCENARIO):
            policy = heuristic_policy_factory()
            r = run_episode(lambda o, p=policy: p(o), sid)
            r['policy'] = 'Heuristic'
            r['episode'] = ep
            r['template_id'] = sid.split('__')[0]
            rows.append(r)
    return rows

heuristic_results = evaluate_heuristic()
print(f'Heuristic mean: {sum(r["final_score"] for r in heuristic_results)/len(heuristic_results):.3f}')

## Policy 3 — Scripted-optimal baseline (reference upper bound on hand-coding)

The scripted-optimal baseline knows exactly what to do per template. It's the upper bound for any non-LLM agent.

In [ ]:
from unified_incident_env.server.challenge import list_baselines, get_scenario

def scripted_optimal_policy_factory(scenario_id):
    bl = list_baselines(scenario_id=scenario_id).baselines[0]
    queue = list(bl.actions)
    state = {}
    def policy(obs):
        if not queue:
            return {'action_type': 'escalate'}
        step = queue.pop(0)
        return step.action.model_dump(exclude_none=True)
    return policy

scripted_results = []
for sid in HOLDOUT_IDS:
    for ep in range(EPISODES_PER_SCENARIO):
        policy = scripted_optimal_policy_factory(sid)
        r = run_episode(lambda o, p=policy: p(o), sid)
        r['policy'] = 'Scripted-optimal'
        r['episode'] = ep
        r['template_id'] = sid.split('__')[0]
        scripted_results.append(r)
print(f'Scripted-optimal mean: {sum(r["final_score"] for r in scripted_results)/len(scripted_results):.3f}')

## Policy 4 — Llama-3.3-70B-Instruct (Fireworks)

Frontier-LLM baseline via Fireworks API. Falls back to Groq if Fireworks key isn't available.


In [ ]:
import openai
FIREWORKS_KEY = os.environ.get('FIREWORKS_API_KEY')
GROQ_KEY = os.environ.get('GROQ_API_KEY')
if FIREWORKS_KEY:
    llama_client = openai.OpenAI(api_key=FIREWORKS_KEY, base_url='https://api.fireworks.ai/inference/v1')
    LLAMA_MODEL = 'accounts/fireworks/models/llama-v3p3-70b-instruct'
elif GROQ_KEY:
    llama_client = openai.OpenAI(api_key=GROQ_KEY, base_url='https://api.groq.com/openai/v1')
    LLAMA_MODEL = 'llama-3.3-70b-versatile'
else:
    llama_client = None
    LLAMA_MODEL = None

def llama_policy(obs):
    if llama_client is None:
        return None
    resp = llama_client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': obs.prompt_text},
        ],
        temperature=0.0,
        max_tokens=200,
    )
    return parse_action(resp.choices[0].message.content)

if llama_client is not None:
    llama_results = evaluate_policy(f'Llama-3.3-70B ({LLAMA_MODEL.split("/")[-1]})', llama_policy)
else:
    llama_results = []
    print('Skipping Llama — no FIREWORKS_API_KEY or GROQ_API_KEY')

## Policy 5+6 — Claude Haiku + Sonnet (Anthropic)

The trained 3B's *direct* targets — these are the rows that have to be beaten.

In [ ]:
import anthropic
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')

def claude_policy_factory(model_id):
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    def policy(obs):
        if not ANTHROPIC_KEY:
            return None
        msg = client.messages.create(
            model=model_id,
            max_tokens=200,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': obs.prompt_text}],
        )
        text = ''.join(b.text for b in msg.content if b.type == 'text')
        return parse_action(text)
    return policy

if ANTHROPIC_KEY:
    haiku_results  = evaluate_policy('Claude Haiku 4.5',  claude_policy_factory('claude-haiku-4-5-20251001'))
    sonnet_results = evaluate_policy('Claude Sonnet 4.6', claude_policy_factory('claude-sonnet-4-6'))
else:
    haiku_results = sonnet_results = []
    print('Skipping Claude — no ANTHROPIC_API_KEY')

## Policy 7 — Trained Qwen2.5-3B (sre-gym-qwen25-3b-grpo)

Loads the trained adapter you produced in `01_basic_train_grpo_unsloth.ipynb` (or pulls it from HF). This is the row that proves the pitch.

In [ ]:
from unsloth import FastLanguageModel
import torch

TRAINED_REPO = 'dakshdoesdev/sre-gym-qwen25-3b-grpo'
LOCAL_TRAINED = Path('/content/grpo-out/lora')

trained_model = trained_tokenizer = None
if LOCAL_TRAINED.exists() or 'HF_TOKEN' in os.environ:
    src = str(LOCAL_TRAINED) if LOCAL_TRAINED.exists() else TRAINED_REPO
    trained_model, trained_tokenizer = FastLanguageModel.from_pretrained(
        model_name=src, max_seq_length=4096, load_in_4bit=True, dtype=None,
    )
    FastLanguageModel.for_inference(trained_model)

def trained_policy(obs):
    if trained_model is None:
        return None
    text = trained_tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': obs.prompt_text}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = trained_tokenizer(text, return_tensors='pt').to(trained_model.device)
    out = trained_model.generate(**inputs, max_new_tokens=128, do_sample=False,
                                  pad_token_id=trained_tokenizer.eos_token_id)
    completion = trained_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_action(completion)

if trained_model is not None:
    trained_results = evaluate_policy('Trained Qwen2.5-3B (GRPO)', trained_policy)
else:
    trained_results = []
    print('Trained adapter not found — run 01_basic_train_grpo_unsloth.ipynb first or set HF_TOKEN.')

## Aggregate, plot, persist

Per-template reward distributions on shared axes; a CSV summary; and the comparison table the README cites.

In [ ]:
all_rows = (random_results + heuristic_results + scripted_results +
            llama_results + haiku_results + sonnet_results + trained_results)
df = pd.DataFrame([{**r['breakdown'], 'final_score': r['final_score'],
                    'resolved': r['incident_resolved'], 'ticks': r['tick_count'],
                    'policy': r['policy'], 'template_id': r['template_id'],
                    'scenario_id': r['scenario_id']} for r in all_rows])
print(df.head())

# Save raw
out_dir = REPO_ROOT / 'eval' / 'results'
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'comparison_raw.csv', index=False)

# Summary table
summary = df.groupby('policy').agg(
    episodes=('final_score','count'),
    resolved=('resolved', 'sum'),
    mean_score=('final_score', 'mean'),
    median_score=('final_score', 'median'),
    p25=('final_score', lambda s: s.quantile(0.25)),
    p75=('final_score', lambda s: s.quantile(0.75)),
).round(3).sort_values('mean_score')
summary.to_csv(out_dir / 'comparison_summary.csv')
print(summary)

In [ ]:
# Plot per-template reward distributions on shared axes.
import math
templates = sorted(df['template_id'].unique())
n = len(templates)
cols = 4
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), sharey=True)
axes = axes.flatten()
for i, tid in enumerate(templates):
    sub = df[df['template_id'] == tid]
    sub.boxplot(column='final_score', by='policy', ax=axes[i], rot=30)
    axes[i].set_title(tid, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('final_score')
    axes[i].set_ylim(0, 1.0)
    axes[i].axhline(0.80, color='red', linestyle='--', alpha=0.4, linewidth=1)
for j in range(n, len(axes)):
    axes[j].axis('off')
plt.suptitle('Reward distribution per template (held-out scenarios)\nDashed line: hardened scripted-optimal ceiling 0.80', y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'comparison_per_template.png', dpi=120, bbox_inches='tight')
print(f'Saved -> {out_dir / "comparison_per_template.png"}')
plt.show()

In [ ]:
# Single-axis comparison plot — the README hero figure.
fig, ax = plt.subplots(figsize=(10, 4.5))
policies = list(summary.index)
means = summary['mean_score'].values
p25 = summary['p25'].values
p75 = summary['p75'].values
errs = [means - p25, p75 - means]
colors = ['#888' if 'Trained' not in p else '#1565c0' for p in policies]
ax.barh(range(len(policies)), means, xerr=errs, color=colors, alpha=0.85)
ax.set_yticks(range(len(policies)))
ax.set_yticklabels(policies)
ax.axvline(0.80, color='red', linestyle='--', linewidth=1.5, label='Scripted-optimal ceiling (0.80)')
ax.set_xlabel('mean reward (held-out, 36 episodes per policy)')
ax.set_xlim(0, 1.0)
ax.legend(loc='lower right')
ax.set_title('sre-gym Basic tier — held-out reward across policies')
plt.tight_layout()
plt.savefig(out_dir / 'comparison_hero.png', dpi=140, bbox_inches='tight')
print(f'Saved -> {out_dir / "comparison_hero.png"}')
plt.show()

In [ ]:
# Final printable comparison table for the README.
tbl = summary.reset_index()[['policy','episodes','resolved','mean_score','median_score']].rename(columns={
    'mean_score': 'mean',
    'median_score': 'median',
})
print(tbl.to_string(index=False))
tbl.to_csv(out_dir / 'comparison_table.csv', index=False)
print(f'\nSaved -> {out_dir / "comparison_table.csv"}')

## Interpreting the comparison

If the trained 3B mean lands above Claude Haiku, **the 20% "reward improvement" judging weight is defended cleanly**. If it lands above Sonnet on at least one template, the 40% "environment innovation" weight is also defended.

Any single trained-3B episode that hits >0.85 is in the headroom band that only a trained agent can reach — the scripted-optimal baseline tops out at ~0.77. That's the clearest possible evidence that **training on this env produced a measurable improvement**.

## Reproducibility

Every artifact this notebook produces lands in `eval/results/` and is committed to the repo:

- `comparison_raw.csv`     — every per-episode row
- `comparison_summary.csv` — per-policy aggregates
- `comparison_table.csv`   — printable comparison table
- `comparison_per_template.png`
- `comparison_hero.png`    — the README hero figure